# Knowledge Graph Construction
## What is a knowledge graph?
The graph has two interconnected layers:

### 1. Lexical Subgraph (Document Structure)
This models the physical structure of the source papers. It preserves provenance: every extracted fact traces back to a specific text passage.
-  *Document*: One node per paper (so 879 in total). It holds metadata.
 *Chunk*: One node per text chunk (31,704 total). Holds the text content and links to its parent Document.
- *Relationships:*
  - `PART_OF`: Chunk -> Document (which paper does this chunk belong to?)
  - `NEXT_CHUNK`: Chunk -> Chunk (reading order within a paper)

### 2. Domain Subgraph (Scientific Knowledge)
Models the actual biological entities and their relationships, extracted by an LLM from the chunk text.

- *AlgalTaxon*: Species, genera, families (e.g., *Ulva pertusa*, *Porphyra*, Rhodophyta)
- *ChemicalCompound*: Metabolites, pigments, toxins, polysaccharides (e.g., fucoxanthin, carrageenan)
- *BiologicalProcess*: Physiological or ecological events (e.g., photosynthesis, harmful algal bloom)
- *Environment*: Habitats, biomes, geographic locations (e.g., intertidal zone, Yellow Sea)
- *ExperimentalParameter*: Measured/manipulated variables (e.g., salinity at 20 psu, temperature)
- *GeneticSequence*: Molecular markers used in phylogenetics (e.g., rbcL, ITS, SSU)
- *Method*: Techniques and procedures (e.g., ddPCR, HPLC, morphological identification)
- *Application*: Industrial or practical uses (e.g., biofuel production, nutraceuticals)


### 3. The Bridge: MENTIONS
The `MENTIONS` relationship connects the two subgraphs. When a Chunk contains text about *Porphyra birdiae*, I create:
**
(Chunk)-[:MENTIONS]->(AlgalTaxon {scientific_name: "Porphyra birdiae"})
**
This is what makes the graph useful for RAG: we can traverse from a domain concept back to the exact source text that supports it.


## Why This Structure?

1. **Traceability** — Every domain node connects back to the chunks that mention it, and those chunks connect to their source documents. The LLM can cite its sources.

2. **Multi-hop reasoning** — Questions like "What methods are used to study HAB-causing species?" require traversing: Method → AlgalTaxon → BiologicalProcess. Vector search can't do this.

3. **Entity unification** — When 50 papers mention *Ulva pertusa*, they all link to one node. The graph aggregates knowledge across the corpus.

## Node metadata code

In [ ]:
# for node metadata
def filename_to_doi(filename):
    # algae-2002-17-4-203.pdf -> 10.4490/algae.2002.17.4.203
    stem = filename.replace(".pdf", "").replace(".json", "")
    parts = stem.split("-")  # ["algae", "2002", "17", "4", "203"]
    return f"10.4490/{parts[0]}.{parts[1]}.{parts[2]}.{parts[3]}.{parts[4]}"

## Deepseek api calls

In [ ]:
import os
from openai import OpenAI
# i need to add money
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": "..."}]
)

APIStatusError: Error code: 402 - {'error': {'message': 'Insufficient Balance', 'type': 'unknown_error', 'param': None, 'code': 'invalid_request_error'}}

In [4]:
def extract_with_cache(chunk_id, chunk_text, cache_dir="data/kg_extractions"):
    cache_path = os.path.join(cache_dir, f"{chunk_id}.json")
    
    # Already extracted? Return cached result
    if os.path.exists(cache_path):
        with open(cache_path, "r") as f:
            return json.load(f)
    
    # Call API
    result = call_deepseek(chunk_text)
    
    # Cache it
    os.makedirs(cache_dir, exist_ok=True)
    with open(cache_path, "w") as f:
        json.dump(result, f)
    
    return result